# DATA PROCESSING


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime
import re

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, acf, pacf, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.linear_model import LinearRegression

import joblib as jb

from google.colab import drive
drive.mount('/content/drive')

pd.options.display.float_format = '{:,.2f}'.format

import warnings
warnings.filterwarnings('ignore')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
folder = '/content/drive/MyDrive/datagodataexplorers2026/RawData/'
df0 = {}
df0['HopDong'] = pd.read_csv(f"{folder}Hợp Đồng.csv")
df0['DonHang'] = pd.read_csv(f"{folder}Đơn Hàng.csv")
df0['XepHangKH'] = pd.read_csv(f"{folder}Xếp Hạng Khách Hàng.csv")
df0['XeCoGioi'] = pd.read_csv(f"{folder}Xe Cơ Giới.csv")
df0['DoTuoi'] = pd.read_csv(f"{folder}Độ Tuổi.csv")
df0['TrangThai']    = pd.read_csv(f"{folder}Trạng Thái Đăng Ký.csv")
df0['ThongTinXe']   = pd.read_csv(f"{folder}Thông Tin Xe Cơ Giới.csv")
df0['PhongBan']     = pd.read_csv(f"{folder}Phòng Ban.csv")
df0['NhomSanPham']  = pd.read_csv(f"{folder}Nhóm Sản Phẩm.csv")
df0['NhanVien']     = pd.read_csv(f"{folder}Nhân Viên.csv")
df0['NgheNghiep']   = pd.read_csv(f"{folder}Nghề Nghiệp.csv")
df0['MaNhanVien']   = pd.read_csv(f"{folder}Mã Nhân Viên.csv")
df0['LoaiXe']       = pd.read_csv(f"{folder}Loại Xe Cơ Giới.csv")
df0['KhachHang']    = pd.read_csv(f"{folder}Khách Hàng.csv")
df0['KenhBanCT']    = pd.read_csv(f"{folder}Kênh Bán Chi Tiết.csv")
df0['KenhBan']      = pd.read_csv(f"{folder}Kênh Bán.csv")
df0['HangXe']       = pd.read_csv(f"{folder}Hãng Xe.csv")
df0['GoiSanPham']   = pd.read_csv(f"{folder}Gói Sản Phẩm.csv")
df0['CongTy']       = pd.read_csv(f"{folder}Công Ty.csv")
df0['ChiNhanh']     = pd.read_csv(f"{folder}Chi Nhánh.csv")
df0['Kpi']          = pd.read_csv(f"{folder}KPI.csv")
df0['BaoHiem']      = pd.read_csv(f"{folder}Bảo Hiểm.csv")
for key, item in df0.items():
    print(f"Dữ liệu bảng: {key}")
    print(item.head())


Dữ liệu bảng: HopDong
          MA_HD MA_GSP     MA_KH
0  XCG_D0000251  GSP01  KH000251
1  XCG_D0000252  GSP01  KH000252
2  XCG_D0000253  GSP01  KH000253
3  XCG_D0000254  GSP01  KH000254
4  XCG_D0000255  GSP01  KH000255
Dữ liệu bảng: DonHang
         MA_HD         MA_DON LOAI_HD      NGAY_KY_HD   NGAY_HIEU_LUC  \
0  CN_D0114049  CN_DBH0000057     Mới  6/10/2025 0:00  6/11/2025 0:00   
1  CN_D0114070  CN_DBH0000078     Mới   2/9/2023 0:00  2/11/2023 0:00   
2  CN_D0114077  CN_DBH0000085     Mới  3/13/2023 0:00  3/16/2023 0:00   
3  CN_D0117949  CN_DBH0003957     Mới  12/7/2024 0:00  12/8/2024 0:00   
4  CN_D0117959  CN_DBH0003967     Mới  7/30/2024 0:00   8/1/2024 0:00   

     NGAY_HET_HAN     MA_KH       GIA_BH          MA_NV MA_KENHBANCHITIET  \
0  6/11/2026 0:00  Kh370116 1,000,000.00   MGV-QN-CN498              DC23   
1  2/11/2024 0:00  Kh391925 1,000,000.00   MGV-CT-CN035              DC19   
2  3/16/2024 0:00  Kh299875 1,000,000.00   MGV-CT-CN175              DC19   
3  12/8/202

In [ ]:
for key, item in df0.items():
    print(f"Dữ liệu bảng: {key}")
    print(item.tail())

Dữ liệu bảng: HopDong
               MA_HD MA_GSP     MA_KH
150489  XCG_D0026496  GSP03  KH026496
150490  XCG_D0026497  GSP03  KH026497
150491  XCG_D0026498  GSP03  KH026498
150492  XCG_D0026499  GSP03  KH026499
150493  XCG_D0026500  GSP03  KH026500
Dữ liệu bảng: DonHang
               MA_HD          MA_DON  LOAI_HD      NGAY_KY_HD   NGAY_HIEU_LUC  \
189773  XCG_D0023365  XCG_DBH0073077  Tái tục  3/28/2025 0:00  3/29/2025 0:00   
189774  XCG_D0022576  XCG_DBH0070818  Tái tục  4/26/2025 0:00  4/27/2025 0:00   
189775  XCG_D0022348  XCG_DBH0072621  Tái tục  3/19/2025 0:00  3/20/2025 0:00   
189776  XCG_D0021672  XCG_DBH0072196  Tái tục  4/28/2025 0:00  4/29/2025 0:00   
189777  XCG_D0022351  XCG_DBH0075855  Tái tục  5/30/2025 0:00  5/31/2025 0:00   

          NGAY_HET_HAN     MA_KH       GIA_BH           MA_NV  \
189773  3/29/2026 0:00  KH023365 1,404,000.00  TQP-TCT-XCG001   
189774  4/27/2026 0:00  KH022576 1,404,000.00  TQP-TCT-XCG001   
189775  3/20/2026 0:00  KH022348 1,404,000.00 

In [ ]:
for key, item in df0.items():
    print(f"\n===========Dữ liệu bảng: {key}================")
    print(item.info())


===========Dữ liệu bảng: HopDong================
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150494 entries, 0 to 150493
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   MA_HD   150494 non-null  object
 1   MA_GSP  150494 non-null  object
 2   MA_KH   150494 non-null  object
dtypes: object(3)
memory usage: 3.4+ MB
None

===========Dữ liệu bảng: DonHang================
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 189778 entries, 0 to 189777
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   MA_HD              189778 non-null  object 
 1   MA_DON             189778 non-null  object 
 2   LOAI_HD            189778 non-null  object 
 3   NGAY_KY_HD         189778 non-null  object 
 4   NGAY_HIEU_LUC      189778 non-null  object 
 5   NGAY_HET_HAN       189778 non-null  object 
 6   MA_KH              189778 non-null  object 
 7   GIA

**3 BẢNG FACT:**
- Bảng đơn hàng: ghi nhận chi tiết từng giao dịch, phí bảo hiểm, thời hạn, kết nối với khách hàng, nhân viên, gói sp, kênh bán
- bảng hợp đồng: quản lý cấp độ hợp đồng
- bảng kpi: tổng hợp doanh thu theo tháng, theo phòng ban, kênh bán, gói sản phẩm

**Bảng DIM:**
- khách hàng
- nhân sự
- sản phẩm
- tài sản bảo hiểm (xe cơ giới)

**Bảng CẤU TRÚC TỔ CHỨC:**
- cơ cấu công ty: Công ty -> chi nhánh -> phòng ban
- kênh phân phối: kênh bán -> kênh bán Chi tiết

**CÁC LỖI:**
**1. Lỗi khoảng trắng**: bảng xe cơ giới (Giá tiền), kpi (doanh thu)

**2. Sai lệch kiểu dữ liệu:**
- Ngày tháng: bảng đơn hàng: ngày ký hd, ngày hiệu lực, ngày hết hạn (object - mm/dd/yyyy). **-> ép về datetime**
- Tiền tệ: giá BH, phí BH, doanh thu, giá tiền (xe) đang dính dấu , và thập phân .00 & chuỗi object **-> xóa ký tự và ép về float hoặc int**
- số điện thoại nhân viên: bị hiểu thành float -> ép về chuỗi & thêm số 0 đầu
- lỗi chính tả danh mục: bảng độ tuổi:** 55 trở nên -> 55 trở lên**

**3. dữ liệu khuyết thiếu:**
- bảng nhân viên:
  + 602 dòng nhưng bảng tuổi có 600 dòng: thiếu 2 dòng ở cột tuổi và sđt
  + Tuổi nhân viên đang để dạng float **-> làm tròn và ép int**

In [ ]:

#KIỂM TRA MISSING, DUPLICATE DÒNG & KHÓA CHÍNH
# Khai báo trước Khóa chính của các bảng quan trọng để test
primary_keys = {
    'HopDong': 'MA_HD', 'DonHang': 'MA_DON', 'KhachHang': 'MA_KH',
    'NhanVien': 'MA_GOCNV', 'XeCoGioi': 'MA_DONG_XE', 'GoiSanPham': 'MA_GOISANPHAM'
}

print("\n KIỂM TRA MISSING, DUPLICATE DÒNG & KHÓA CHÍNH:")
for name, df in df0.items():
    # 1. Missing Values
    missing = df.isnull().sum() #đếm số lượng ô trống ở từng cột
    cols_with_missing = missing[missing > 0] #lọc ra những cột có số lượng rỗng >0

    # 2. Duplicate Dòng
    dup_rows = df.duplicated().sum() #các dòng giống nhau

    # In kết quả nếu có lỗi
    if not cols_with_missing.empty or dup_rows > 0 or name in primary_keys:
        print(f" \n--- Bảng [{name}] ---")
        if not cols_with_missing.empty:
            print(f"  * Khuyết thiếu: {cols_with_missing.to_dict()}")
        if dup_rows > 0:
            print(f"  * Trùng lặp dòng: {dup_rows} dòng")

        # 3. Duplicate Khóa chính
        if name in primary_keys and primary_keys[name] in df.columns:
            pk = primary_keys[name]
            dup_pk = df.duplicated(subset=[pk]).sum()
            if dup_pk > 0:
                print(f"Trùng lặp Khóa chính ({pk}): {dup_pk} mã bị trùng!")

# ---------------------------------------------------------
# BƯỚC 4: KIỂM TRA TOÀN VẸN THAM CHIẾU (MÃ MỒ CÔI)
# ---------------------------------------------------------
print("\n KIỂM TRA TOÀN VẸN THAM CHIẾU (MÃ MỒ CÔI):")

orphan_kh = ~df0['DonHang']['MA_KH'].isin(df0['KhachHang']['MA_KH'])
print(f" * Số đơn hàng chứa MA_KH không tồn tại trong bảng KhachHang: {orphan_kh.sum()} đơn")

orphan_gsp = ~df0['DonHang']['MA_GSP'].isin(df0['GoiSanPham']['MA_GOISANPHAM'])
print(f" * Số đơn hàng chứa MA_GSP không tồn tại trong bảng GoiSanPham: {orphan_gsp.sum()} đơn")

orphan_nv = ~df0['DonHang']['MA_NV'].isin(df0['MaNhanVien']['MA_NV'])
print(f" * Số đơn hàng chứa MA_NV không tồn tại trong hệ thống (MaNhanVien): {orphan_nv.sum()} đơn")

# ---------------------------------------------------------
# BƯỚC 5: KIỂM TRA TÍNH ĐỒNG NHẤT CHUỖI (LỖI KHOẢNG TRẮNG CỘT)
# ---------------------------------------------------------
print("\n[III] KIỂM TRA TÊN CỘT BỊ DƯ KHOẢNG TRẮNG:")

for name, df in df0.items():
    bad_cols = [col for col in df.columns if col.startswith(' ') or col.endswith(' ')]
    if bad_cols:
        print(f" * Bảng [{name}] có cột sai định dạng: {bad_cols}")

# ---------------------------------------------------------
# BƯỚC 6 & 7: KIỂM TRA NGHIỆP VỤ & GIÁ TRỊ NGOẠI LAI (OUTLIERS)
# ---------------------------------------------------------
print("\n KIỂM TRA NGHIỆP VỤ & NGOẠI LAI:")

# 6. Logic Thời Gian (Dùng errors='coerce' để vượt qua các dòng lỗi format)
temp_hieu_luc = pd.to_datetime(df0['DonHang']['NGAY_HIEU_LUC'], format='mixed', errors='coerce')
temp_het_han = pd.to_datetime(df0['DonHang']['NGAY_HET_HAN'], format='mixed', errors='coerce')
logic_error_mask = temp_het_han <= temp_hieu_luc
print(f" - Số đơn hàng sai logic (Ngày Hết Hạn <= Ngày Hiệu Lực): {logic_error_mask.sum()} đơn")

# 7.1. Giá trị Doanh thu/Phí bảo hiểm vô lý (Ép kiểu chuỗi tiền tệ tạm thời để check)
temp_phibh = df0['DonHang']['PHIBH'].astype(str).str.replace(',', '').str.replace('.00', '', regex=False).astype(float)
invalid_money = temp_phibh <= 0
print(f" - Số đơn hàng có Phí bảo hiểm bị âm hoặc bằng 0: {invalid_money.sum()} đơn")

# 7.2. Tuổi vô lý
invalid_age = (df0['KhachHang']['TUOI'] < 18) | (df0['KhachHang']['TUOI'] > 110)
print(f" - Số khách hàng có độ tuổi vô lý (<18 hoặc >110): {invalid_age.sum()} khách hàng")



 KIỂM TRA MISSING, DUPLICATE DÒNG & KHÓA CHÍNH:
 
--- Bảng [HopDong] ---
 
--- Bảng [DonHang] ---
 
--- Bảng [XeCoGioi] ---
 
--- Bảng [NhanVien] ---
  * Khuyết thiếu: {'TUOI': 2, 'SDT': 2}
 
--- Bảng [KhachHang] ---
 
--- Bảng [GoiSanPham] ---
 
--- Bảng [Kpi] ---
  * Trùng lặp dòng: 682 dòng

 KIỂM TRA TOÀN VẸN THAM CHIẾU (MÃ MỒ CÔI):
 * Số đơn hàng chứa MA_KH không tồn tại trong bảng KhachHang: 0 đơn
 * Số đơn hàng chứa MA_GSP không tồn tại trong bảng GoiSanPham: 0 đơn
 * Số đơn hàng chứa MA_NV không tồn tại trong hệ thống (MaNhanVien): 0 đơn

[III] KIỂM TRA TÊN CỘT BỊ DƯ KHOẢNG TRẮNG:
 * Bảng [XeCoGioi] có cột sai định dạng: [' GIA_TIEN ']
 * Bảng [Kpi] có cột sai định dạng: [' DOANH_THU ']

 KIỂM TRA NGHIỆP VỤ & NGOẠI LAI:
 - Số đơn hàng sai logic (Ngày Hết Hạn <= Ngày Hiệu Lực): 0 đơn
 - Số đơn hàng có Phí bảo hiểm bị âm hoặc bằng 0: 0 đơn
 - Số khách hàng có độ tuổi vô lý (<18 hoặc >110): 0 khách hàng


**XỬ LÝ DỮ LIỆU**

In [ ]:
# LÀM SẠCH DỮ LIỆU

# 1. SỬA LỖI TÊN CỘT (KHOẢNG TRẮNG)
# Áp dụng hàm strip() cho tên cột của TOÀN BỘ 22 bảng để chắc chắn không bỏ sót
for key in df0.keys():
    df0[key].columns = df0[key].columns.str.strip()
print(" 1. Đã xóa toàn bộ khoảng trắng ẩn trong tên cột.")

# ==========================================
# 2: XÓA DỮ LIỆU TRÙNG LẶP (DUPLICATES)
# Bảng KPI bị trùng 682 dòng, ta sẽ giữ lại dòng đầu tiên và xóa các dòng lặp
df0['Kpi'].drop_duplicates(keep='first', inplace=True)
print(" 2. Đã xóa 682 dòng trùng lặp trong bảng KPI.")

# ==========================================
# BƯỚC 3: SỬA LỖI CHÍNH TẢ DANH MỤC

df0['DoTuoi']['NHOM_TUOI'] = df0['DoTuoi']['NHOM_TUOI'].str.replace('Trở nên', 'Trở lên')
print(" 3. Đã sửa lỗi chính tả '55 Trở nên' thành '55 Trở lên'.")

# ==========================================
# BƯỚC 4: ÉP KIỂU NGÀY THÁNG VÀ TIỀN TỆ

# 4.1. Ngày tháng (Bảng DonHang)
date_cols = ['NGAY_KY_HD', 'NGAY_HIEU_LUC', 'NGAY_HET_HAN']
for col in date_cols:
    df0['DonHang'][col] = pd.to_datetime(df0['DonHang'][col], format='mixed', errors='coerce')

# 4.2. Tiền tệ (Bảng DonHang, Kpi, XeCoGioi)
# Hàm phụ trợ làm sạch chuỗi tiền tệ
def clean_currency(series):
    # Đổi sang chuỗi -> xóa dấu phẩy -> xóa đuôi .00 -> ép về Float
    return series.astype(str).str.replace(',', '').str.replace('.00', '', regex=False).astype(float)

df0['DonHang']['GIA_BH'] = clean_currency(df0['DonHang']['GIA_BH'])
df0['DonHang']['PHIBH'] = clean_currency(df0['DonHang']['PHIBH'])
df0['Kpi']['DOANH_THU'] = clean_currency(df0['Kpi']['DOANH_THU'])
df0['XeCoGioi']['GIA_TIEN'] = clean_currency(df0['XeCoGioi']['GIA_TIEN'])
print(" 4. Đã ép kiểu Datetime và chuẩn hóa thành công các cột Tiền tệ về kiểu số Thực (Float).")

# ==========================================
# BƯỚC 5: XỬ LÝ DỮ LIỆU KHUYẾT THIẾU BẢNG NHÂN VIÊN

# 5.1. Cột Tuổi: Điền 2 ô trống bằng tuổi trung vị (median) của nhóm, sau đó làm tròn và ép về số nguyên (Int)
median_age = df0['NhanVien']['TUOI'].median()
df0['NhanVien']['TUOI'] = df0['NhanVien']['TUOI'].fillna(median_age).round().astype(int)

# 5.2. Cột SĐT: Hàm xử lý float -> string và khôi phục số 0
def fix_phone_number(val):
    if pd.isna(val):
        return 'Chưa cập nhật' # Hoặc gán 'Unknown' cho 2 người bị thiếu
    else:
        # Ép về int để mất đuôi .0, ép về chuỗi, thêm '0' ở đầu
        return '0' + str(int(val))

df0['NhanVien']['SDT'] = df0['NhanVien']['SDT'].apply(fix_phone_number)
print(" 5. Đã xử lý khuyết thiếu tuổi/sđt và ép tuổi Nhân viên về chuẩn Int, SĐT về chuỗi có số 0 ở đầu.")



 1. Đã xóa toàn bộ khoảng trắng ẩn trong tên cột.
 2. Đã xóa 682 dòng trùng lặp trong bảng KPI.
 3. Đã sửa lỗi chính tả '55 Trở nên' thành '55 Trở lên'.
 4. Đã ép kiểu Datetime và chuẩn hóa thành công các cột Tiền tệ về kiểu số Thực (Float).
 5. Đã xử lý khuyết thiếu tuổi/sđt và ép tuổi Nhân viên về chuẩn Int, SĐT về chuỗi có số 0 ở đầu.


In [ ]:
print("="*50)
print("BÁO CÁO NGHIỆM THU DỮ LIỆU SAU LÀM SẠCH")
print("="*50)

# 1. KIỂM TRA TỔNG QUAN MISSING & DUPLICATE
total_missing = sum(df.isnull().sum().sum() for df in df0.values())
total_duplicates = sum(df.duplicated().sum() for df in df0.values())

print("\n[1] TÌNH TRẠNG RÁC DỮ LIỆU TỔNG THỂ:")
print(f" - Tổng số ô khuyết thiếu (Missing) trên 22 bảng: {total_missing}")
print(f" - Tổng số dòng trùng lặp (Duplicates) trên 22 bảng: {total_duplicates}")

# 2. KIỂM TRA TÊN CỘT
bad_cols = [col for df in df0.values() for col in df.columns if col.startswith(' ') or col.endswith(' ')]
print(f"\n[2] TÌNH TRẠNG TÊN CỘT:")
print(f" - Số lượng tên cột còn dính khoảng trắng: {len(bad_cols)}")
if bad_cols:
    print(f"   Chi tiết: {bad_cols}")

# 3. KIỂM TRA LỖI CHÍNH TẢ ĐẶC THÙ
typo_count = (df0['DoTuoi']['NHOM_TUOI'] == '55 Trở nên').sum()
print(f"\n[3] KIỂM TRA LỖI CHÍNH TẢ:")
print(f" - Số dòng còn lỗi '55 Trở nên' trong bảng DoTuoi: {typo_count}")

# 4. KIỂM TRA ÉP KIỂU (DATA TYPES) CỦA CÁC CỘT TRỌNG ĐIỂM
print("\n[4] KIỂM TRA KIỂU DỮ LIỆU TRỌNG ĐIỂM:")

# Hàm phụ trợ in kết quả Đạt/Chưa đạt
def check_type(col_name, current_type, expected_type_keyword):
    status = "✅ ĐẠT" if expected_type_keyword in str(current_type).lower() else "❌ LỖI"
    # Convert current_type to string before formatting to avoid TypeError
    print(f" - {col_name:<25}: {str(current_type):<15} [{status}]")

# Kiểm tra Ngày tháng
check_type("NGAY_KY_HD (DonHang)", df0['DonHang']['NGAY_KY_HD'].dtype, "datetime")
# Kiểm tra Tiền tệ
check_type("PHIBH (DonHang)", df0['DonHang']['PHIBH'].dtype, "float")
check_type("DOANH_THU (Kpi)", df0['Kpi']['DOANH_THU'].dtype, "float")
check_type("GIA_TIEN (XeCoGioi)", df0['XeCoGioi']['GIA_TIEN'].dtype, "float")
# Kiểm tra Tuổi & SĐT Nhân viên
check_type("TUOI (NhanVien)", df0['NhanVien']['TUOI'].dtype, "int")
check_type("SDT (NhanVien)", df0['NhanVien']['SDT'].dtype, "object") # String trong Pandas gọi là object

print("\n" + "="*50)

BÁO CÁO NGHIỆM THU DỮ LIỆU SAU LÀM SẠCH

[1] TÌNH TRẠNG RÁC DỮ LIỆU TỔNG THỂ:
 - Tổng số ô khuyết thiếu (Missing) trên 22 bảng: 0
 - Tổng số dòng trùng lặp (Duplicates) trên 22 bảng: 0

[2] TÌNH TRẠNG TÊN CỘT:
 - Số lượng tên cột còn dính khoảng trắng: 0

[3] KIỂM TRA LỖI CHÍNH TẢ:
 - Số dòng còn lỗi '55 Trở nên' trong bảng DoTuoi: 0

[4] KIỂM TRA KIỂU DỮ LIỆU TRỌNG ĐIỂM:
 - NGAY_KY_HD (DonHang)     : datetime64[ns]  [✅ ĐẠT]
 - PHIBH (DonHang)          : float64         [✅ ĐẠT]
 - DOANH_THU (Kpi)          : float64         [✅ ĐẠT]
 - GIA_TIEN (XeCoGioi)      : float64         [✅ ĐẠT]
 - TUOI (NhanVien)          : int64           [✅ ĐẠT]
 - SDT (NhanVien)           : object          [✅ ĐẠT]



In [ ]:
import os

# 1. Tạo một thư mục mới để chứa dữ liệu Sạch (tránh ghi đè lên dữ liệu gốc)
# Bạn có thể đổi tên thư mục này tùy ý
output_folder = '/content/drive/MyDrive/datagodataexplorers2026/Data_Cleaned/'

# Tạo thư mục nếu nó chưa tồn tại
os.makedirs(output_folder, exist_ok=True)

print("BẮT ĐẦU XUẤT FILE CSV...")
print("-" * 40)

# 2. Vòng lặp xuất từng bảng trong df0 ra file CSV
for table_name, df in df0.items():
    # Tạo tên file (VD: HopDong_Cleaned.csv)
    file_name = f"{table_name}_Cleaned.csv"
    file_path = os.path.join(output_folder, file_name)

    # Xuất file CSV
    # index=False: Không xuất cột số thứ tự (0, 1, 2...)
    # encoding='utf-8-sig': Chống lỗi font Tiếng Việt khi mở bằng Excel
    df.to_csv(file_path, index=False, encoding='utf-8-sig')

    print(f"✅ Đã xuất thành công: {file_name} ({len(df)} dòng)")

print("-" * 40)
print(f"HOÀN TẤT! Toàn bộ 22 file đã được lưu tại: {output_folder}")

BẮT ĐẦU XUẤT FILE CSV...
----------------------------------------
✅ Đã xuất thành công: HopDong_Cleaned.csv (150494 dòng)
✅ Đã xuất thành công: DonHang_Cleaned.csv (189778 dòng)
✅ Đã xuất thành công: XepHangKH_Cleaned.csv (5 dòng)
✅ Đã xuất thành công: XeCoGioi_Cleaned.csv (263 dòng)
✅ Đã xuất thành công: DoTuoi_Cleaned.csv (100 dòng)
✅ Đã xuất thành công: TrangThai_Cleaned.csv (5 dòng)
✅ Đã xuất thành công: ThongTinXe_Cleaned.csv (64486 dòng)
✅ Đã xuất thành công: PhongBan_Cleaned.csv (68 dòng)
✅ Đã xuất thành công: NhomSanPham_Cleaned.csv (9 dòng)
✅ Đã xuất thành công: NhanVien_Cleaned.csv (602 dòng)
✅ Đã xuất thành công: NgheNghiep_Cleaned.csv (32 dòng)
✅ Đã xuất thành công: MaNhanVien_Cleaned.csv (1122 dòng)
✅ Đã xuất thành công: LoaiXe_Cleaned.csv (36 dòng)
✅ Đã xuất thành công: KhachHang_Cleaned.csv (400001 dòng)
✅ Đã xuất thành công: KenhBanCT_Cleaned.csv (97 dòng)
✅ Đã xuất thành công: KenhBan_Cleaned.csv (7 dòng)
✅ Đã xuất thành công: HangXe_Cleaned.csv (51 dòng)
✅ Đã xuất thà